In [1]:
import f3dasm


import sys
import pandas as pd
import tensorflow as tf
tf.get_logger().setLevel('ERROR') # only show error messages

from recommenders.utils.timer import Timer
from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.datasets import movielens
from recommenders.utils.notebook_utils import is_jupyter
from recommenders.datasets.python_splitters import python_chrono_split
from recommenders.evaluation.python_evaluation import (rmse, mae, rsquared, exp_var, map_at_k, ndcg_at_k, precision_at_k, 
                                                     recall_at_k, get_top_k_items)

print("System version: {}".format(sys.version))
print("Pandas version: {}".format(pd.__version__))
print("Tensorflow version: {}".format(tf.__version__))



2023-02-20 21:45:06.745086: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-02-20 21:45:06.912016: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-02-20 21:45:07.596354: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda-11.1/lib64:/usr/local/cuda-11.1/lib64
2023-02-20 21:45:07.596413: W ten

warning in stationary: failed to import cython module: falling back to numpy
warning in coregionalize: failed to import cython module: falling back to numpy
warning in choleskies: failed to import cython module: falling back to numpy
System version: 3.8.16 | packaged by conda-forge | (default, Feb  1 2023, 16:01:55) 
[GCC 11.3.0]
Pandas version: 1.5.3
Tensorflow version: 2.11.0


In [ ]:
# top k items to recommend
TOP_K = 10

# Select MovieLens data size: 100k, 1m, 10m, or 20m
MOVIELENS_DATA_SIZE = '100k'

# Model parameters
EPOCHS = 50
BATCH_SIZE = 256

SEED = 42

In [2]:
df = movielens.load_pandas_df(
    size=MOVIELENS_DATA_SIZE,
    header=["userID", "itemID", "rating", "timestamp"]
)

df


2023-02-20 21:45:10,842 - Downloading https://files.grouplens.org/datasets/movielens/ml-100k.zip
100%|██████████| 4.81k/4.81k [00:04<00:00, 1.17kKB/s]


,userID,itemID,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596
...,...,...,...,...
99995,880,476,3.0,880175444
99996,716,204,5.0,879795543
99997,276,1090,1.0,874795795
99998,13,225,2.0,882399156


In [5]:
train, test = python_chrono_split(df, 0.75)
train, test



(       userID  itemID  rating  timestamp
 59972       1     168     5.0  874965478
 92487       1     172     5.0  874965478
 74577       1     165     5.0  874965518
 48214       1     156     4.0  874965556
 22971       1     166     5.0  874965677
 ...       ...     ...     ...        ...
 77956     943      94     4.0  888639929
 99832     943     739     4.0  888639929
 90134     943       2     5.0  888639953
 87419     943     541     4.0  888639954
 76415     943      27     4.0  888639954
 
 [74992 rows x 4 columns],
        userID  itemID  rating  timestamp
 73358       1     149     2.0  878542791
 88893       1      88     4.0  878542791
 59055       1     101     2.0  878542845
 43438       1     110     1.0  878542845
 14792       1     103     1.0  878542845
 ...       ...     ...     ...        ...
 81005     943     450     1.0  888693158
 92536     943     227     1.0  888693158
 95003     943     230     1.0  888693158
 94914     943     229     2.0  888693158
 9288